Filtering data from yfinance

In [2]:
import pandas as pd
import warnings

warnings.filterwarnings(
    "ignore", category=FutureWarning, message=".*un-recognized timezone.*"
)


Getting data from yfinance starting from the dates that we have gotten from the news data

In [48]:
import yfinance as yf

start_date = "2017-12-17"
end_date = "2020-07-18"

raw_data = yf.download(tickers = "^GSPC", start = start_date,
                              end = end_date, interval = "1d")

# flattening multi-level columns
raw_data.columns = [
    "_".join([str(c) for c in col if c != ""])
    if isinstance(col, tuple)
    else col
    for col in raw_data.columns
]

raw_data.head()

C:\Users\Wiltj\AppData\Local\Temp\ipykernel_17036\1794419794.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw_data = yf.download(tickers = "^GSPC", start = start_date,
[*********************100%***********************]  1 of 1 completed


,Close_^GSPC,High_^GSPC,Low_^GSPC,Open_^GSPC,Volume_^GSPC
Date,,,,,
2017-12-18,2690.159912,2694.969971,2685.919922,2685.919922,3727770000
2017-12-19,2681.469971,2694.439941,2680.739990,2692.709961,3407680000
2017-12-20,2679.250000,2691.010010,2676.110107,2688.179932,3246230000
2017-12-21,2684.570068,2692.639893,2682.399902,2683.020020,3293130000
2017-12-22,2683.340088,2685.350098,2678.129883,2684.219971,2401030000


In [49]:
raw_data.info()
# there is no null data in the dataset
raw_data.isnull().sum()
# nothing to filter but this is where we should grab data that is ready to be used
save_path = f'data/filtered/yfinance_data.csv'
raw_data.to_csv(save_path)
# creating a copy and saving it for better name clarity
df_stock = raw_data.copy()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 649 entries, 2017-12-18 to 2020-07-17
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Close_^GSPC   649 non-null    float64
 1   High_^GSPC    649 non-null    float64
 2   Low_^GSPC     649 non-null    float64
 3   Open_^GSPC    649 non-null    float64
 4   Volume_^GSPC  649 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 30.4 KB


Close is the closing price of the day for the stock  
High is the highest traded price of the day for the stock  
Low is the lowest traded price of the day for the stock  
Open is the opening price of the day for the stock  
Volume is the number of shares traded during the day for the stock 

Using this information and I am going to use it to quantify percent change if the stock went up or down each day and binary classifying the data

In [51]:
# calculating percent change for closing price
df_stock['pct_change'] = df_stock['Close_^GSPC'].pct_change()

# creating target variable: 1 if next day's pct_change > 0 else 0
df_stock['Target'] = (df_stock['pct_change'] > 0).astype(int)

# selecting relevant columns for our testing and training
df_stock_refined = df_stock[['Volume_^GSPC', 'pct_change', 'Target']].copy()

#removing the first row with NaN pct_change
df_stock_refined = df_stock_refined.iloc[1:].reset_index(drop=True)

df_stock_refined.head()

,Volume_^GSPC,pct_change,Target
0,3407680000,-0.003230,0
1,3246230000,-0.000828,0
2,3293130000,0.001986,1
3,2401030000,-0.000458,0
4,1970660000,-0.001058,0
